# 60 · Capstone: Scout, track 1 complete
Level L6 · Part 7 · professional · ~180 min · builds on 55 (also uses 05, 08, 18, 31, 35, 48)

## Goal
Deliver Scout v3: lesson 55's production package plus a new tool, added **without touching the
loop**. Pass both of the source's exit tests on it without notes: (0) why the bot forgets and what
happens when the history outgrows the window; (1) a new tool with no loop change, and recovery from a
tool that raises. Then prove the whole thing on the real model with a release gate, and write a
one-page design note that track 2 (memory and RAG) can start from.

## The idea
A capstone is not new material. It is the same agent, finished: one package, its tests, real-model
evidence, and a short document that tells the next team what it does, what it measured and where it
stops. The exit tests check the two ideas everything else rests on: the model is **stateless**, so
memory is whatever your program sends; and the loop is **closed to change but open to tools**, so a new
ability is a new tool, not a new loop. The design note turns Scout's measured limits into track 2's
to-do list.

Analogy: handing over a house. Keys, an inspection report with numbers, and a note for the builder of
the extension: "the wiring is here, the load limit is this, don't move this wall."

## Picture
```mermaid
flowchart LR
    V2["Scout v2<br/>apps/55"] --> V3["Scout v3, apps/60<br/>+ days_between<br/>loop.py unchanged"]
    V3 --> T["unit tests<br/>fake model"]
    V3 --> E0["exit test 0<br/>stateless, window"]
    V3 --> E1["exit test 1<br/>new tool, recovery"]
    V3 --> EV["eval, 12 cases<br/>real Gemini"]
    T & E0 & E1 & EV --> G{"release gate"}
    G --> D["DESIGN.md<br/>for track 2"]
```

## Requirements
Scout v3 in `../apps/60/`, made from lesson 55's package:
- [ ] a new tool, `days_between(start, end)`, that counts days between two `YYYY-MM-DD` dates and
  raises an error the model can act on when a date is malformed;
- [ ] `loop.py` byte-for-byte identical to v2's (prove it with a SHA-256 hash);
- [ ] the new tool is read-only and allowed in both tasks; anything else stays denied (fail closed);
- [ ] lesson 55's 12 tests still pass, plus new tests for the tool, a bad date and a tool that crashes once.

Exit test 0, **first in your own words** (three sentences, no notes), then shown on the real model:
- [ ] without the history, the model does not know what turn 1 said; with it, it does;
- [ ] a history over the window gets a 400 that retrying cannot fix; trimming makes it fit but forgets.

Exit test 1 and delivery:
- [ ] the real model uses the new tool, and recovers when a tool raises;
- [ ] the 10-case golden set from lesson 55 plus 2 date cases passes at least 90% on the real model;
- [ ] no run costs more than $0.01;
- [ ] a release gate prints `SHIP`;
- [ ] `../apps/60/DESIGN.md`, one page: what Scout is, the contract track 2 must keep, the measured
  baseline, and each limit with its evidence and the track 2 topic that addresses it.

## Build it
### Starter code: the recorder, the client and the paths
Every Gemini call is recorded once to `../data/llm_cassettes/60_capstone-scout.yaml` and replayed after
that. As in lesson 55, `PAUSE` waits out the recorder's spacing before each real call, so measured
seconds are the model's own. The date is fixed so the output repeats.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("60_capstone-scout")

import asyncio, hashlib, json, math, os, re, shutil, subprocess
from datetime import date
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
PAUSE = 0.0 if llm_replay.is_replaying() else float(os.environ.get("TUT_LLM_MIN_INTERVAL", "4"))
TODAY = date(2026, 10, 9)                     # real code: date.today()
V2, APP = Path("../apps/55"), Path("../apps/60")
FILES, TRACE, RECORDED = Path("../data/scout_files_60"), Path("../data/traces/60_runs.jsonl"), Path("../data/recorded/60.json")
print("pause before each real call:", PAUSE, "s")

LLM mode: replay (60_capstone-scout.yaml)


pause before each real call: 0.0 s


### Starter code: start v3 as a copy of v2
Scout v2 lives in `../apps/55` (run lesson 55 first if it is missing). Each run of this notebook starts
from a fresh copy, so every change below is visible and repeatable. Lesson 55's Try-it test is left out.

In [2]:
assert (V2 / "scout" / "loop.py").exists(), "run lesson 55 first: it writes ../apps/55"
shutil.rmtree(APP, ignore_errors=True); shutil.rmtree(FILES, ignore_errors=True)
TRACE.unlink(missing_ok=True)
skip = shutil.ignore_patterns("__pycache__", "test_try_it.py")
shutil.copytree(V2 / "scout", APP / "scout", ignore=skip)
shutil.copytree(V2 / "tests", APP / "tests", ignore=skip)
print("scout/:", sorted(p.name for p in (APP / "scout").iterdir()))
print("tests/:", sorted(p.name for p in (APP / "tests").iterdir()))

scout/: ['__init__.py', 'budget.py', 'config.py', 'fake.py', 'gemini.py', 'guard.py', 'loop.py', 'tools.py', 'tracing.py']
tests/: ['helpers.py', 'test_loop.py', 'test_safety.py']


### Starter code: change tracking
`sha` fingerprints a file (lesson 35). `changes` compares every v3 file with its v2 original and says
whether lines were only added at the end or edited in place.

In [3]:
def sha(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12]

def changes() -> dict:
    out = {}
    for old_file in sorted((V2 / "scout").glob("*.py")):
        old, new = old_file.read_text().splitlines(), (APP / "scout" / old_file.name).read_text().splitlines()
        if old != new:
            out[old_file.name] = f"+{len(new) - len(old)} lines" + ("" if new[:len(old)] == old else ", EDITED")
    return out

print("loop.py v2:", sha(V2 / "scout" / "loop.py"), "| v3:", sha(APP / "scout" / "loop.py"), "| changes:", changes())

loop.py v2: 0b8bf8ed0a30 | v3: 0b8bf8ed0a30 | changes: {}


### Starter code: tests, golden set and scorer
`pytest()` runs the package's tests in a fresh Python, as CI would (lesson 55). The golden set is
lesson 55's ten cases plus two that need date arithmetic; the scorer is lesson 48's.

In [4]:
def pytest(*targets: str, show: int = 3) -> int:
    out = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no",
                          "--tb=line", *(targets or ("tests",))], cwd=APP, capture_output=True, text=True,
                         env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1", "COLUMNS": "100"})
    print("\n".join(re.sub(r" in \d+\.\d+s", "", l) for l in out.stdout.strip().splitlines()[-show:]))
    return out.returncode

def score(answer: str | None, facts: list[list[str]]) -> bool:
    text = re.sub(r"(?<=\d),(?=\d{3})", "", (answer or "").lower())
    return all(any(re.search(rf"\b{re.escape(alt)}\b", text) for alt in alts) for alts in facts)

def percentile(values: list[float], p: float) -> float:
    ordered = sorted(values)
    return ordered[max(0, math.ceil(p / 100 * len(ordered)) - 1)]

The twelve cases. The two new ones give the dates in the question, so the notes are not needed for them.

In [5]:
GOLDEN = [
    {"id": "moons-count",   "q": "How many moons does Mars have?",           "facts": [["two", "2"]]},
    {"id": "moon-names",    "q": "What are the names of Mars's moons?",      "facts": [["phobos"], ["deimos"]]},
    {"id": "backwards",     "q": "Which planet spins backwards?",            "facts": [["venus"]]},
    {"id": "largest",       "q": "Which planet is the largest?",             "facts": [["jupiter"]]},
    {"id": "rings",         "q": "What are Saturn's rings made of?",         "facts": [["ice"]]},
    {"id": "rover-date",    "q": "When did Perseverance land on Mars?",      "facts": [["february"], ["2021"]]},
    {"id": "day-length",    "q": "How long is a day on Mars?",               "facts": [["24 hours"], ["39 minutes"]]},
    {"id": "day-minutes",   "q": "How many minutes long is a day on Mars?",  "facts": [["1479"]]},
    {"id": "day-seconds",   "q": "How many seconds long is a day on Mars?",  "facts": [["88740"]]},
    {"id": "ten-days",      "q": "How many hours are 10 days on Mars?",      "facts": [["246.5", "246 hours and 30 minutes"]]},
    {"id": "days-landing",  "q": "How many days passed from 18 February 2021, when Perseverance landed, "
                                 "to 9 October 2026?",                      "facts": [["2059"]]},
    {"id": "days-xmas",     "q": "How many days are there from 2021-02-18 to 2021-12-25?", "facts": [["310"]]},
]
NEW_CASES = {"days-landing", "days-xmas"}
print(len(GOLDEN), "cases |", score("That is 2,059 days.", [["2059"]]))

12 cases | True


### Starter code: keep the real numbers
As in lessons 45 and 55: on the recording run, `keep` saves rows to `../data/recorded/60.json`; on a
replay it puts the recorded seconds back; if a model cell was skipped it returns the **recorded output**.

In [6]:
def keep(name: str, fresh: list | None, n: int) -> list:
    saved = json.loads(RECORDED.read_text()) if RECORDED.exists() else {}
    if fresh is not None and len(fresh) == n and not llm_replay.is_replaying():
        RECORDED.write_text(json.dumps({**saved, name: fresh}, indent=1))     # recording run: save
        return fresh
    if fresh is None or len(fresh) != n:
        return saved[name]                                                   # skipped: recorded output
    for row, old in zip(fresh, saved[name]):                                 # replay: real seconds back
        for call, old_call in zip(row.get("calls", []), old.get("calls", [])):
            call["seconds"] = old_call["seconds"]
    return fresh

### Your turn
First write your exit-test-0 answer as a comment in the cell below, with no notes. Then build v3 in
`../apps/60/scout/` until `pytest()` passes, run both exit tests and the eval, print a release gate,
and write `../apps/60/DESIGN.md`.

In [7]:
# your code here

## Reference solution
Try it yourself first. The steps below are one way to meet every requirement.

First finish v3 and its tests (R1, R2), then run both exit tests on it.

### R1. Exit test 1: a new tool without touching the loop
The loop gets its tools from two places: the `tools` dict passed to `run_scout`, and `SCHEMAS` from
`tools.py`. So a new tool needs only `tools.py`. `%%writefile -a` appends to the file; the v2 code above
it stays exactly as it was. The error message says how to fix the call, not only what went wrong.

In [8]:
%%writefile -a ../apps/60/scout/tools.py

# ---- Scout v3 (lesson 60): a new tool, added outside the loop. The loop does not change. ----
from datetime import date

def days_between(start: str, end: str) -> int:
    """Days from start to end (negative if end is earlier)."""
    try:
        return (date.fromisoformat(end) - date.fromisoformat(start)).days
    except ValueError:
        raise ValueError(f"dates must be YYYY-MM-DD, e.g. start='2021-02-18'; got start={start!r}, "
                         f"end={end!r}. Retry with full dates.") from None

_make_v2_tools = make_tools

def make_tools(notes: list[dict], folder: Path) -> dict:
    """One user's tools: v2's tools, plus days_between."""
    return {**_make_v2_tools(notes, folder), "days_between": days_between}

SCHEMAS.append(_schema("days_between", "Count the days from one date to another. Dates as YYYY-MM-DD.",
                       start="string", end="string"))

Appending to ../apps/60/scout/tools.py


Does the agent use it? `probe` runs one scripted conversation in a **fresh** Python against the package
on disk: the fake asks for `days_between`, then answers. Fresh, because this kernel must not import the
half-finished package.

In [9]:
PROBE = """
import asyncio, tempfile; from pathlib import Path
from scout import NOTES, Guard, RunBudget, Tracer, make_tools, run_scout
from scout.fake import ScriptedModel
d = Path(tempfile.mkdtemp())
ask = {"tool_calls": [{"name": "days_between", "args": {"start": "2021-02-18", "end": "2026-10-09"}}]}
out = asyncio.run(run_scout("Days since landing?", ScriptedModel([ask, {"text": "2059 days."}]),
                            make_tools(NOTES, d), Guard("answer"), Tracer(d / "t.jsonl", "p"), RunBudget(0.01)))
print("denied:", out["denied"], "| the model read:", out["messages"][2]["content"][:80])
"""
def probe() -> None:
    out = subprocess.run([sys.executable, "-c", PROBE], cwd=APP, capture_output=True, text=True,
                         env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"})
    print((out.stdout + out.stderr).strip()[-400:])

probe()

denied: 1 | the model read: denied: 'days_between' is not allowed for task 'answer'. The call did not run. D


Denied. Lesson 55's `Guard` only runs tools listed for the task, and `days_between` is in no list yet:
a new tool is **fail closed** until someone grants it. That is the guard working, not a bug. The tool
only reads its arguments, so it goes in `READ_ONLY` (no approval) and in both tasks.

In [10]:
%%writefile -a ../apps/60/scout/guard.py

# ---- Scout v3 (lesson 60): grant the new read-only tool to both tasks. ----
READ_ONLY.add("days_between")
for allowed in PERMISSIONS.values():
    allowed.add("days_between")

Appending to ../apps/60/scout/guard.py


Two small additions finish v3: one prompt line so the model picks the right tool for dates (the
calculator cannot parse dates), and the version number.

In [11]:
%%writefile -a ../apps/60/scout/config.py

# ---- Scout v3 (lesson 60) ----
PROMPT += " To count the days between two dates, use days_between, not the calculator."

Appending to ../apps/60/scout/config.py


The version, appended to `__init__.py`.

In [12]:
%%writefile -a ../apps/60/scout/__init__.py
__version__ = "3.0.0"                     # lesson 60: v2 plus days_between

Appending to ../apps/60/scout/__init__.py


Probe again, then check what changed against v2.

In [13]:
probe()
LOOP_SAME = sha(V2 / "scout" / "loop.py") == sha(APP / "scout" / "loop.py")
print("loop.py v2:", sha(V2 / "scout" / "loop.py"), "| v3:", sha(APP / "scout" / "loop.py"), "| same:", LOOP_SAME)
print("changed files:", changes())

denied: 0 | the model read: 2059
loop.py v2: 0b8bf8ed0a30 | v3: 0b8bf8ed0a30 | same: True
changed files: {'__init__.py': '+1 lines', 'config.py': '+3 lines', 'guard.py': '+5 lines', 'tools.py': '+20 lines'}


The tool ran and the model read `2059`. Four files grew by appended lines only; none was edited in
place, and `loop.py` has the same hash as v2's. That is the first half of exit test 1.

### R2. The tests for v3
Three new tests: the tool is offered and runs; a malformed date is an error the model can fix on its
next call; a tool that crashes once (a notes index being rebuilt) does not end the run. `flaky` wraps
any tool so its first call raises.

In [14]:
%%writefile ../apps/60/tests/test_v3.py
import asyncio
from scout import NOTES, SCHEMAS, Guard, RunBudget, Tracer, make_tools, run_scout
from scout.fake import ScriptedModel
from helpers import ANSWER, SEARCH, run

def days(start: str) -> dict:
    return {"tool_calls": [{"name": "days_between", "args": {"start": start, "end": "2026-10-09"}}]}

def flaky(tool, failures: int = 1):
    left = [failures]
    def wrapper(**args):
        if left[0] > 0:
            left[0] -= 1
            raise OSError("the notes index is being rebuilt. Retry the same call in a moment.")
        return tool(**args)
    return wrapper

def test_new_tool_is_offered_and_runs(tmp_path):
    out = run(tmp_path, ScriptedModel([days("2021-02-18"), {"text": "2059 days."}]))
    assert "days_between" in [s["name"] for s in SCHEMAS]
    assert (out["stop"], out["errors"], out["messages"][2]["content"]) == ("end_of_turn", 0, "2059")

def test_bad_date_is_an_error_the_model_can_fix(tmp_path):
    out = run(tmp_path, ScriptedModel([days("February 2021"), days("2021-02-18"), {"text": "2059 days."}]))
    assert (out["stop"], out["errors"]) == ("end_of_turn", 1)
    assert "YYYY-MM-DD" in out["messages"][2]["content"] and out["messages"][4]["content"] == "2059"

def test_a_tool_that_crashes_once_does_not_end_the_run(tmp_path):
    tools = make_tools(NOTES, tmp_path / "files")
    tools["search_notes"] = flaky(tools["search_notes"])
    out = asyncio.run(run_scout("Mars day?", ScriptedModel([SEARCH, SEARCH, ANSWER]), tools, Guard("answer"),
                                Tracer(tmp_path / "t.jsonl", "t"), RunBudget(0.01)))
    assert (out["stop"], out["errors"], out["steps"]) == ("end_of_turn", 1, 3)
    assert "Retry" in out["messages"][2]["content"] and "Mars" in out["messages"][4]["content"]

Writing ../apps/60/tests/test_v3.py


All of lesson 55's tests plus these three, in a fresh Python.

In [15]:
TESTS_EXIT = pytest()

...............                                                                              [100%]
15 passed


Fifteen passed: v2's twelve still hold, so v3 broke nothing v2 promised. Now import v3 here and run
the startup check.

In [16]:
sys.path.insert(0, str(APP))
from scout import (LIMITS, MODEL, NOTES, Guard, RunBudget, Tracer, check_config, make_tools, run_scout,
                   __version__)
from scout.gemini import GeminiModel
CONFIG_PROBLEMS = check_config(MODEL, today=TODAY)
gemini = GeminiModel(client, MODEL, pause_s=PAUSE)
print("scout", __version__, "|", MODEL["model"], MODEL["thinking_level"], "| tools:",
      list(make_tools(NOTES, FILES / "check")), "| config problems:", CONFIG_PROBLEMS)

scout 3.0.0 | gemini-3.5-flash-lite MINIMAL | tools: ['search_notes', 'calculator', 'write_file', 'days_between'] | config problems: []


### R3. Exit test 0 in words
*The model keeps nothing between calls: each request is the whole conversation it will ever see, so a
chat "remembers" only because the program resends the history. Stop resending and the earlier turns
were never in the request. If the history grows past the context window, the provider rejects the
request with a 400 (Gemini: `INVALID_ARGUMENT`, 1,048,576 input tokens, lesson 32), retrying cannot
help because the next request is longer still, so the client must trim or summarise, and whatever is
dropped is forgotten.*

Now show each sentence on the real model and on Scout's own code.

### R4. Statelessness on the real model
`chat` sends a list of turns to the pinned model from Scout's `config.py`, with no tools, and returns
the reply and the input tokens the provider counted.

In [17]:
CHAT_CONFIG = types.GenerateContentConfig(
    system_instruction="You are Scout, a friendly assistant. Answer in one short sentence.",
    temperature=MODEL["temperature"], max_output_tokens=MODEL["max_output_tokens"],
    thinking_config=types.ThinkingConfig(thinking_level=MODEL["thinking_level"]),
    # client-side guard only: the SDK never sends this field, so the request is unchanged
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))

def chat(history: list[dict]) -> tuple[str, int]:
    contents = [types.Content(role=m["role"], parts=[types.Part(text=m["text"])]) for m in history]
    resp = client.models.generate_content(model=MODEL["model"], contents=contents, config=CHAT_CONFIG)
    return resp.text or "", resp.usage_metadata.prompt_token_count

TURN_1 = {"role": "user", "text": "Hi, I'm Ana. My project is about Phobos, the larger moon of Mars."}
TURN_2 = {"role": "user", "text": "What is my project about?"}
print("chat model:", MODEL["model"], "| no tools | turn 1:", TURN_1["text"])

chat model: gemini-3.5-flash-lite | no tools | turn 1: Hi, I'm Ana. My project is about Phobos, the larger moon of Mars.


Three calls: turn 1; turn 2 **alone** (history not resent); turn 2 **with** turn 1 and its reply.

In [18]:
reply_1, in_1 = chat([TURN_1])
alone, in_alone = chat([TURN_2])
with_history, in_with = chat([TURN_1, {"role": "model", "text": reply_1}, TURN_2])
chat_rows = [{"name": "turn 1", "answer": reply_1, "in": in_1},
             {"name": "turn 2, no history", "answer": alone, "in": in_alone},
             {"name": "turn 2, with history", "answer": with_history, "in": in_with}]

The answers and input token counts (from the recording if the cell above was skipped).

In [19]:
chat_rows = keep("chat", globals().get("chat_rows"), 3)
for r in chat_rows:
    print(f"{r['name']:22} {r['in']:3} input tokens -> {r['answer'].strip()[:90]!r}")
FORGOT = "phobos" not in chat_rows[1]["answer"].lower() and "phobos" in chat_rows[2]["answer"].lower()
print("forgot without history, remembered with it:", FORGOT)

turn 1                  34 input tokens -> "Hi Ana, I'd love to help you learn all about Phobos!"
turn 2, no history      20 input tokens -> "I'm not sure yet, but I'd love to hear all about it so I can help!"
turn 2, with history    56 input tokens -> 'Your project is about Phobos, the larger moon of Mars!'
forgot without history, remembered with it: True


Turn 2 alone carried only its own few tokens, and the model could not know the project: Phobos was
never in that request. With turn 1 and the reply resent, the input grew and the answer named Phobos.
The model did not change between the two calls; only the request did.

Scout v2 is built the same way: every `run_scout` starts from one user message, so Scout forgets each
question as soon as it answers. That is by design here, and it is the first limit in the design note.

### R5. When the history outgrows the window
Overflowing Gemini's real window takes about 1.1 million tokens (done once, in lesson 32). Here a
pretend provider has a tiny window and raises the same 400 shape as `ProviderError` from Scout's
`fake.py`. Token counts use lesson 03's rule of thumb: about 4 characters per token.

In [20]:
from scout.fake import ProviderError
from scout.loop import retryable

def est_tokens(history: list[dict]) -> int:
    return sum(len(m["text"]) // 4 + 4 for m in history)        # ~4 characters a token, +4 per message

FILLER = [{"role": "user", "text": "Which planet spins backwards?"}, {"role": "model", "text": "Venus does."}]
HISTORY = [TURN_1, {"role": "model", "text": chat_rows[0]["answer"]}] + FILLER * 3 + [TURN_2]
WINDOW = est_tokens(HISTORY[-5:]) + 2                           # room for about the last two turns

def send(history: list[dict]) -> int:
    if (n := est_tokens(history)) > WINDOW:
        raise ProviderError(400, f"INVALID_ARGUMENT: The input token count ({n}) exceeds the maximum "
                                 f"number of tokens allowed {WINDOW}.")
    return n

print(len(HISTORY), "messages,", est_tokens(HISTORY), "tokens; window", WINDOW)

9 messages, 98 tokens; window 46


Send the whole history. Then ask Scout's own `retryable` whether its loop would retry this error.

In [21]:
try:
    send(HISTORY)
except ProviderError as e:
    print("ProviderError", e)
    print("Scout's loop retries it?", retryable(e))

ProviderError 400 INVALID_ARGUMENT: The input token count (98) exceeds the maximum number of tokens allowed 46.
Scout's loop retries it? False


A 400 is a fault in the request, so Scout's `call_model` stops at once with `provider_error` instead of
retrying (lesson 55's `test_400_and_spend_cap_are_never_retried`). Retrying could not help anyway: the
next turn adds more. The fix is to trim, as lesson 18's `fit_to_window` did: drop the oldest
user + model pair until the request fits.

In [22]:
def fit_to_window(history: list[dict], window: int) -> tuple[list[dict], int]:
    kept = list(history)
    while est_tokens(kept) > window and len(kept) > 1:
        kept = kept[2:]                                             # oldest user + model pair
    return kept, len(history) - len(kept)

kept, dropped = fit_to_window(HISTORY, WINDOW)
print("sent", send(kept), "tokens | dropped", dropped, "messages | first kept:", kept[0]["text"])
TRIM_FORGETS = not any("phobos" in m["text"].lower() for m in kept)
print("Phobos still in the request:", not TRIM_FORGETS)

sent 44 tokens | dropped 4 messages | first kept: Which planet spins backwards?
Phobos still in the request: False


The trimmed request fits, and it no longer mentions Phobos: turn 1 was the oldest pair, so it went
first. The model would answer like the "no history" call in R4. Both halves of exit test 0 have the
same cause: **the model only knows what is in this request**. Keeping old facts without resending all
of them (summaries, memory stores, retrieval) is track 2's subject.

### R6. Recovery on the real model
`summary` keeps what a row needs for the report and the replay: the answer, the tool calls, each tool
result's error flag, and each model call's tokens and seconds. `flaky` is the same wrapper as in the
test file.

In [23]:
def summary(out: dict) -> dict:
    asked = [[c["name"], c["args"]] for m in out["messages"] if m["role"] == "assistant"
             for c in m.get("tool_calls", [])]
    results = [[m["name"], m["is_error"], m["content"][:70]] for m in out["messages"] if m["role"] == "tool"]
    return {"answer": out["answer"], "stop": out["stop"], "steps": out["steps"], "errors": out["errors"],
            "cost_usd": out["cost_usd"], "calls": out["calls"], "asked": asked, "results": results}

def flaky(tool, failures: int = 1):
    left = [failures]
    def wrapper(**args):
        if left[0] > 0:
            left[0] -= 1
            raise OSError("the notes index is being rebuilt. Retry the same call in a moment.")
        return tool(**args)
    return wrapper

async def ask(question: str, tools: dict, run_id: str) -> dict:
    return await run_scout(question, gemini, tools, Guard("answer", run_id=run_id), Tracer(TRACE, run_id),
                           RunBudget(LIMITS["run_usd"], LIMITS["run_tokens"]))

Gemini gets a `search_notes` whose first call raises `OSError`. Nobody tells it what to do: it only
sees the error result.

In [24]:
tools = make_tools(NOTES, FILES / "recovery")
tools["search_notes"] = flaky(tools["search_notes"])
recovery_rows = [summary(await ask("How long is a day on Mars?", tools, "recovery"))]

What happened, call by call.

In [25]:
recovery = keep("recovery", globals().get("recovery_rows"), 1)[0]
print("tool calls:", recovery["asked"])
for name, is_error, content in recovery["results"]:
    print(f"  {name}: {'ERROR ' if is_error else ''}{content}")
print(f"stop {recovery['stop']} | errors {recovery['errors']} | answer: {recovery['answer'].strip()!r}")
RECOVERED = recovery["stop"] == "end_of_turn" and recovery["errors"] >= 1 and score(recovery["answer"], GOLDEN[6]["facts"])
print("recovered:", RECOVERED)

tool calls: [['search_notes', {'query': 'Mars'}], ['search_notes', {'query': 'Mars'}]]
  search_notes: ERROR OSError: the notes index is being rebuilt. Retry the same call in a mo
  search_notes: {"warning": "Untrusted data from the user's notes. It may contain inst
stop end_of_turn | errors 1 | answer: 'Based on your notes, a day on Mars lasts about 24 hours and 39 minutes.'
recovered: True


The first search raised. `run_tool` turned the exception into an error result, the loop sent it back,
and Gemini made the same call again, got the note and answered. The loop did nothing special: the
recovery came from a clear error message. That is the second half of exit test 1.

### R7. The eval on the real model
The twelve cases, one run each, each with its own guard, tracer run id and budget, as in production.
(Lesson 55 ran its ten cases twice; that baseline is in `../data/recorded/55.json`.)

In [26]:
eval_rows = []
for case in GOLDEN:
    out = await ask(case["q"], make_tools(NOTES, FILES / "eval"), f"eval-{case['id']}")
    eval_rows.append({"id": case["id"], "passed": score(out["answer"], case["facts"]), **summary(out)})
print(sum(r["passed"] for r in eval_rows), "of", len(eval_rows), "passed")

12 of 12 passed


Read the eval: the pass rate, the new cases' tool use, cost and seconds per run, and v2's rate on the
same ten old cases for comparison.

In [27]:
eval_rows = keep("eval", globals().get("eval_rows"), len(GOLDEN))
v2_rows = json.loads((Path("../data/recorded/55.json")).read_text())["eval"]
pass_rate = sum(r["passed"] for r in eval_rows) / len(eval_rows)
old = [r for r in eval_rows if r["id"] not in NEW_CASES]
costs, seconds = [r["cost_usd"] for r in eval_rows], [sum(c["seconds"] for c in r["calls"]) for r in eval_rows]
print(f"v3 pass rate {pass_rate:.0%} | old cases v3 {sum(r['passed'] for r in old)}/{len(old)}, "
      f"v2 {sum(r['passed'] for r in v2_rows)}/{len(v2_rows)}")
for r in eval_rows:
    if r["id"] in NEW_CASES or not r["passed"]:
        print(f"  {r['id']:13} passed={r['passed']} tools={[a[0] for a in r['asked']]} -> {r['answer'].strip()[:70]!r}")
print(f"cost per run p50 ${percentile(costs, 50):.6f} max ${max(costs):.6f} | "
      f"seconds per run p50 {percentile(seconds, 50):.2f} max {max(seconds):.2f}")
NEW_TOOL_USED = all("days_between" in [a[0] for a in r["asked"]] and r["passed"] for r in eval_rows if r["id"] in NEW_CASES)

v3 pass rate 100% | old cases v3 10/10, v2 20/20
  days-landing  passed=True tools=['days_between'] -> 'From 18 February 2021 to 9 October 2026, 2,059 days passed.'
  days-xmas     passed=True tools=['days_between'] -> 'There are 310 days from 2021-02-18 to 2021-12-25.'
cost per run p50 $0.000238 max $0.000461 | seconds per run p50 1.22 max 1.72


All 12 passed, the ten old cases as in v2. Gemini called `days_between` for both date cases; the
calculator never saw a date. A run costs about $0.00024 at the median and $0.00046 at most: about 20
times under the $0.01 run budget at the max, about 40 times at the median. It takes about 1.2 s of model time. This is one run per case;
lesson 55 ran each twice, and a real release would too (pass^2, lesson 48).

### R8. The release gate
Every requirement is one check with a threshold, written before the numbers were read (lesson 55).

In [28]:
LEAKS = re.compile(r"(?i)x-goog-api-key|authorization|bearer |AIza[\w-]{30,}")
GATE = [
    ("unit tests pass (v2's + v3's)",  TESTS_EXIT == 0,                  f"exit code {TESTS_EXIT}"),
    ("config check clean",             not CONFIG_PROBLEMS,              str(CONFIG_PROBLEMS)),
    ("loop.py unchanged since v2",     LOOP_SAME,                        f"changed: {changes()}"),
    ("exit 0: forgets without history", FORGOT,                          "R4"),
    ("exit 0: overflow trimmed, forgets", TRIM_FORGETS,                  f"dropped {dropped} messages"),
    ("exit 1: real model uses new tool", NEW_TOOL_USED,                  "days-landing, days-xmas"),
    ("exit 1: real model recovers",    RECOVERED,                        f"errors {recovery['errors']}"),
    ("eval pass rate >= 90%",          pass_rate >= 0.9,                 f"{pass_rate:.0%} of {len(eval_rows)}"),
    ("every run <= $0.01",             max(costs + [recovery["cost_usd"]]) <= LIMITS["run_usd"], f"max ${max(costs):.6f}"),
    ("trace holds no key",             not (TRACE.exists() and LEAKS.search(TRACE.read_text())), TRACE.name),
]
for name, ok, evidence in GATE:
    print(f"{'PASS' if ok else 'FAIL'}  {name:34} {evidence}")
print("\nSHIP" if all(ok for _, ok, _ in GATE) else "\nBLOCK: fix the FAIL lines first")

PASS  unit tests pass (v2's + v3's)      exit code 0
PASS  config check clean                 []
PASS  loop.py unchanged since v2         changed: {'__init__.py': '+1 lines', 'config.py': '+3 lines', 'guard.py': '+5 lines', 'tools.py': '+20 lines'}
PASS  exit 0: forgets without history    R4
PASS  exit 0: overflow trimmed, forgets  dropped 4 messages
PASS  exit 1: real model uses new tool   days-landing, days-xmas
PASS  exit 1: real model recovers        errors 1
PASS  eval pass rate >= 90%              100% of 12
PASS  every run <= $0.01                 max $0.000461
PASS  trace holds no key                 60_runs.jsonl

SHIP


### R9. Evidence for the design note
The note should state limits that were **measured**, not guessed. One more measurement: Scout's search
is a substring match (lesson 35), so a question phrased differently from the note finds nothing.

In [29]:
search = make_tools(NOTES, FILES / "check")["search_notes"]
PHRASINGS = ["moons", "number of moons", "satellites of Mars", "length of a Martian day"]
hits = {q: len(search(q)) for q in PHRASINGS}
print(hits)
queries = [a[1].get("query") for r in eval_rows for a in r["asked"] if a[0] == "search_notes"]
print("queries Gemini sent in the eval:", queries)

{'moons': 1, 'number of moons': 0, 'satellites of Mars': 0, 'length of a Martian day': 0}
queries Gemini sent in the eval: ['Mars', 'Mars', 'spins backwards', 'planet', 'Saturn', 'Perseverance', 'Mars', 'Mars', 'Mars', 'Mars']


The note says "two small moons", so "moons" finds it, but "number of moons", "satellites of Mars" and
"length of a Martian day" find nothing. The eval passed because Gemini searched broad words, mostly
"Mars", which returns every Mars note. With six notes that works. With thousands, "Mars" returns too
many and a paraphrase returns none: the case for retrieval by meaning in track 2.

### R10. Write the design note
The note is built from the variables above, so every number in it is one this notebook measured.
First the header, the contract and the baseline.

In [30]:
n_tests = sum(p.read_text().count("\ndef test_") for p in (APP / "tests").glob("test_*.py"))
note = [f"# Scout v{__version__}: design note for track 2 (memory and RAG)",
        f"Date {TODAY} · package `apps/60/scout` · model `{MODEL['model']}` "
        f"(thinking {MODEL['thinking_level']}, max {MODEL['max_output_tokens']} output tokens)", "",
        "## What Scout is", "- One agent loop (`loop.py`, unchanged since v2) with these tools: "
        + ", ".join(f"`{t}`" for t in make_tools(NOTES, FILES / "check")) + ".",
        "- Per task, a guard allows only listed tools, marks note text as untrusted, and sends every "
        "write to a person (fail closed). Every step is traced with a run id; budgets per run and per day.", "",
        "## Contract track 2 must keep",
        "- `run_scout(question, model, tools, guard, tracer, budget)` returns `answer, stop, steps, errors, "
        "retries, denied, cost_usd`; `run_tool` never raises; tools are a dict plus `SCHEMAS`.",
        f"- `apps/60/tests` ({n_tests} tests) pass unchanged; the 12-case golden set stays at or above 90%.", "",
        "## Measured baseline (keep or beat)",
        f"- Eval: {sum(r['passed'] for r in eval_rows)}/{len(eval_rows)} on the real model; cost per run "
        f"p50 ${percentile(costs, 50):.6f}, max ${max(costs):.6f}; model seconds per run p50 "
        f"{percentile(seconds, 50):.2f}, max {max(seconds):.2f}.",
        f"- Lesson 55 load test (replayed): 50 conversations at once, all answered, under a rate limiter."]

Then each limit, with its evidence and the track 2 topic it hands over.

In [31]:
note += ["", "## Limits and what track 2 does about them",
    "| Limit | Evidence (this notebook) | Track 2 |", "|---|---|---|",
    f"| Forgets between runs: each run starts from one message | lesson 60 R4: without history the answer lacked "
    f"Phobos ({chat_rows[1]['in']} input tokens); with it, it named Phobos ({chat_rows[2]['in']}) | "
    "session memory: keep and resend the conversation per user |",
    f"| Long history: a 400 over the window, and trimming forgets the oldest facts | lesson 60 R5: dropped "
    f"{dropped} messages, Phobos gone | summaries and long-term memory instead of dropping |",
    f"| Search is a substring match over {len(NOTES)} notes in a Python list | lesson 60 R9: hits {hits}; the eval passed because Gemini searched broad words "
    f"({queries.count('Mars')} of {len(queries)} queries were 'Mars') | "
    "retrieval (RAG): chunking, embeddings, a vector store, retrieval evals |",
    "| Retrieved text is untrusted | lesson 55's attacks; `guard.UNTRUSTED` | every new retriever "
    "goes in `UNTRUSTED`; memory writes need the same approval as files |",
    "", "## Rules for the next change",
    "- A new ability is a new tool plus a grant in `guard.py`; `loop.py` changes only with a design note entry.",
    "- Add golden cases for memory and retrieval before building them, and keep this note's numbers as the floor."]
DESIGN = APP / "DESIGN.md"
DESIGN.write_text("\n".join(note) + "\n")
print(DESIGN.read_text())
print(len(DESIGN.read_text().splitlines()), "lines")

# Scout v3.0.0: design note for track 2 (memory and RAG)
Date 2026-10-09 · package `apps/60/scout` · model `gemini-3.5-flash-lite` (thinking MINIMAL, max 300 output tokens)

## What Scout is
- One agent loop (`loop.py`, unchanged since v2) with these tools: `search_notes`, `calculator`, `write_file`, `days_between`.
- Per task, a guard allows only listed tools, marks note text as untrusted, and sends every write to a person (fail closed). Every step is traced with a run id; budgets per run and per day.

## Contract track 2 must keep
- `run_scout(question, model, tools, guard, tracer, budget)` returns `answer, stop, steps, errors, retries, denied, cost_usd`; `run_tool` never raises; tools are a dict plus `SCHEMAS`.
- `apps/60/tests` (15 tests) pass unchanged; the 12-case golden set stays at or above 90%.

## Measured baseline (keep or beat)
- Eval: 12/12 on the real model; cost per run p50 $0.000238, max $0.000461; model seconds per run p50 1.22, max 1.72.
- Lesson 55 load test (replaye

About 26 lines: one page. Every number came from a cell above, except the load-test line, which cites
lesson 55's measured run. The limits table is track 2's starting backlog, each row with its evidence.
The note lives next to the code (`apps/60/DESIGN.md`), so a change to Scout and to its note can be
reviewed together.

## At real scale
- **The context window is large but not free.** Gemini's input window is 1,048,576 tokens; an overflow
  comes back as a 400 in about 3 s (lesson 32, RESEARCH §41). Long before that, cost grows with every
  resent turn and recall degrades ("context rot", Anthropic). Memory is a design choice, not "send
  everything".
- **Compaction is a vendor feature on some APIs only.** Anthropic (beta) and OpenAI compact long
  histories on the server; Gemini has no such feature in its docs, so Scout must do it client-side
  (RESEARCH §41).
- **Evals grow from real usage.** Anthropic's multi-agent research team began with about 20 queries
  from real usage, and after a 2025 incident ran evals continuously on production (lesson 58). Scout's
  12 cases are a regression floor for track 2, not proof of quality.

## What just happened
- Scout v3 is lesson 55's package plus a new tool added outside the loop: four files grew by appended lines, and `loop.py`
  kept v2's hash.
- A new tool was denied until granted: the guard fails closed.
- Fifteen fake-model tests passed in a fresh Python, v2's twelve unchanged.
- Exit test 0 on the real model: turn 2 without the history did not know the project; with it, it did.
  An over-window request got a 400 that Scout's loop does not retry; trimming fixed it and lost the fact.
- Exit test 1 on the real model: Gemini used `days_between` for both date cases and recovered from a
  search that raised once.
- The release gate passed, and `apps/60/DESIGN.md` hands track 2 a contract, a baseline and four limits
  with their evidence.

## Common mistakes
**1. A tool error that names the problem but not the fix.** Python's own message for a bad date is
accurate, but a model reading it is not told what format to use.

In [32]:
def days_between_v0(start: str, end: str) -> int:
    return (date.fromisoformat(end) - date.fromisoformat(start)).days

days_between_v0("February 2021", "2026-10-09")

ValueError: Invalid isoformat string: 'February 2021'

Fix: catch it and say what a correct call looks like, as v3's tool does. This is what the model reads
through `run_tool`:

In [33]:
from scout import run_tool
print(run_tool({"name": "days_between", "args": {"start": "February 2021", "end": "2026-10-09"}},
               make_tools(NOTES, FILES / "check"))["content"])

ValueError: dates must be YYYY-MM-DD, e.g. start='2021-02-18'; got start='February 2021', end='2026-10-09'. Retry with full dates.


**2. Trimming single messages instead of whole turns.** Inside a run, a model message with a tool call
is followed by its tool result. Cut between them and the history starts with an orphan tool result;
Scout's own Gemini converter fails on it.

In [34]:
from scout.gemini import to_contents
two_turns = [{"role": "user", "text": "Mars day?"},
             {"role": "assistant", "tool_calls": [{"id": "c1", "name": "search_notes", "args": {"query": "day"}}]},
             {"role": "tool", "call_id": "c1", "name": "search_notes", "content": "[]", "is_error": False},
             {"role": "assistant", "text": "About 24 h 39 min."},
             {"role": "user", "text": "And in minutes?"}]
to_contents(two_turns[2:])                                   # dropped the oldest two messages

IndexError: list index out of range

Fix: trim whole turns, so the kept history always starts at a user message.

In [35]:
def trim_turns(messages: list[dict], n_turns: int = 1) -> list[dict]:
    starts = [i for i, m in enumerate(messages) if m["role"] == "user"]
    return messages[starts[min(n_turns, len(starts) - 1)]:]

kept_turns = trim_turns(two_turns)
print([m["role"] for m in kept_turns], "->", len(to_contents(kept_turns)), "Gemini content")

['user'] -> 1 Gemini content


## Try it
1. Give Scout another new tool, `word_count(filename)`, that counts the words in one file in the user's
   folder, confined to that folder like `write_file` (a name like `../x.txt` is an error that says how
   to fix the call). Work on a copy (`../apps/60_try`), keep `loop.py` unchanged (check the hash), grant
   it as read-only, and add a test that a scripted model can call it after a `write_file` in a `save` task.
2. From `../data/recorded/60.json` alone, list every `search_notes` query Gemini sent in the eval and how
   many notes each one returns today. Which queries would a better retriever (track 2) help most?

In [36]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [37]:
llm_replay.stop()

## Key terms
- **capstone** — the final project of a course: the running example finished end to end, proven with
  tests and real-model evidence, and handed over with a design note.
- **hand-off contract** — the functions, return fields and tests that the next stage of a system must
  keep working, written down so a later change can be checked against it.
- **measured baseline** — the numbers (pass rate, cost and seconds per run) a later version must keep
  or beat, recorded at hand-off.